# Ch 5. 프롬프트 엔지니어링 + CoT 기초

원본: [AI Assistant Engineering - Part 2 Ch 5](https://desty.github.io/study-ai-assistant-engineering/part2/05-prompt-cot/)

## 이 챕터에서 배우는 것

- 프롬프트는 모델에게 주는 **계약서** 라는 관점
- **시스템 프롬프트**의 5요소 (역할 · 지시 · 제약 · 예시 · 출력 형식)
- **Few-shot**: 예시 몇 개로 복잡한 규칙을 가르치기
- **Chain-of-Thought (CoT)**: "단계별로 생각해" 한 줄이 정확도를 올리는 이유
- "모르면 모른다" 지시로 hallucination 1차 방어
- 프롬프트 **인젝션**과 토큰 과소비 실수

In [ ]:
# 필수 패키지
!pip install -q anthropic

In [ ]:
# API 키 설정
import os
from getpass import getpass

if not os.getenv('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass('ANTHROPIC_API_KEY: ')

In [ ]:
from anthropic import Anthropic
client = Anthropic()

## 4. 최소 예제 — 시스템 프롬프트의 유/무 비교

In [ ]:
question = "오늘 저녁 메뉴 추천해줘"

# 1) 시스템 프롬프트 없이
r1 = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=256,
    messages=[{"role": "user", "content": question}],
)

# 2) 시스템 프롬프트 있음
r2 = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=256,
    system="당신은 500kcal 이하 저칼로리 식단만 제안하는 영양사입니다. 3줄 이내로 답하세요.",  # (1)!
    messages=[{"role": "user", "content": question}],
)

print("--- 1) 지침 없음 ---\n", r1.content[0].text)
print("\n--- 2) 지침 있음 ---\n", r2.content[0].text)

## 5. 실전 튜토리얼

### 5.1 시스템 프롬프트 5요소

In [ ]:
SYSTEM = """
[역할]
당신은 전자상거래 고객 지원 어시스턴트입니다.

[지시]
사용자 문의를 읽고 다음 중 하나로 분류하세요:
- refund (환불)
- shipping (배송)
- product (제품 문의)
- other (기타)

[제약]
- 오직 위 4개 중 하나만 반환
- 분류가 애매하면 "other"
- 이유·인사·부연 설명 금지

[출력 형식]
{"category": "<하나>", "confidence": <0~1 실수>}
"""

print("SYSTEM 프롬프트 템플릿이 정의되었습니다.")

### 5.2 Few-shot — 예시로 가르치기

In [ ]:
SYSTEM = "감정을 positive / negative / neutral 중 하나로 분류하세요. 한 단어만."

history = [
    {"role": "user",      "content": "이 제품 정말 최고예요!"},
    {"role": "assistant", "content": "positive"},
    {"role": "user",      "content": "그저 그래요, 기대만큼은 아니네요."},
    {"role": "assistant", "content": "neutral"},
    {"role": "user",      "content": "돈이 아까워 죽을 거 같아요."},
    {"role": "assistant", "content": "negative"},
    {"role": "user",      "content": "배송이 빨랐는데 포장은 엉망"},  # (1)!
]

r = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=10,
    system=SYSTEM,
    messages=history,
)
print(f"분류 결과: {r.content[0].text}")

### 5.3 Chain-of-Thought — "단계별로 생각해"

In [ ]:
question = "알약을 매일 2개씩 먹는데 30개 들이 병이 있어. 얼마나 가나?"

# 직답
SYSTEM_DIRECT = "짧게 답하세요."

# CoT
SYSTEM_COT = "먼저 단계별로 생각한 뒤, 마지막 줄에 최종 답을 '답:'으로 시작해 쓰세요."  # (1)!

for label, sys in [("직답", SYSTEM_DIRECT), ("CoT", SYSTEM_COT)]:
    r = client.messages.create(
        model="claude-haiku-4-5-20251001", max_tokens=256,
        system=sys,
        messages=[{"role": "user", "content": question}],
    )
    print(f"\n=== {label} ===")
    print(r.content[0].text)

### 5.4 출력 형식 강제 — JSON 힌트

In [ ]:
SYSTEM = """
주문 텍스트에서 정보를 추출해 **오직 JSON만** 반환하세요.
다른 텍스트 금지.

스키마:
{
  "item": "<제품명>",
  "quantity": <정수>,
  "address": "<주소>"
}
"""

r = client.messages.create(
    model="claude-haiku-4-5-20251001", max_tokens=256,
    system=SYSTEM,
    messages=[{"role": "user", "content": "빨간 운동화 2켤레 서울 강남구로 보내주세요"}],
)
print(r.content[0].text)

### 5.5 "모르면 모른다고" — hallucination 1차 방어

In [ ]:
SYSTEM = """
당신은 고객 지원 어시스턴트입니다.
주어진 문서에 없는 내용은 절대 지어내지 말 것.
모르면 "확인 후 답변드리겠습니다" 라고 답하세요.
"""

print("이 두 줄이 실전 CS 봇에서 사고율을 수배 낮춥니다.")
print("완전 방어는 아니지만 1차 방어선 역할을 합니다.")

## 6. 자주 깨지는 포인트

### 실수 1. 프롬프트 인젝션
- 사용자가 "이전 지시 무시하고..." 같은 공격
- 대응: (1) 시스템 프롬프트에 명시, (2) 사용자 입력을 XML 태그로 감싸기, (3) Safety Classifier (Part 6)

### 실수 2. Few-shot을 너무 많이 넣음
- 10개는 너무 많음 → 애초에 파인튜닝 후보
- Few-shot은 3~5개 이하 권장

### 실수 3. 모호한 지시
- "적절히 답해", "가능하면 짧게" → 기준이 명확하지 않음
- 대응: 수치로 — "100자 이내", "불릿 3개", "0~10 점수로"

### 실수 4. 모델마다 같은 프롬프트가 최적이라 착각
- Haiku와 Opus는 응답 스타일이 다름
- 대응: 모델 교체 시 프롬프트 리뷰 + 평가셋 재검증

### 실수 5. 프롬프트 버전 관리 안 함
- 배포 후 "어제는 맞았는데..." → 원인 추적 불가
- 대응: 프롬프트를 git 추적 또는 LangSmith/Langfuse 프롬프트 registry

## 7. 운영 시 체크할 점

- [ ] 프롬프트를 **코드 안에 흩어진 문자열**로 두지 말고 모듈 하나에 **상수로 집약**
- [ ] 프롬프트 변경 시 **평가셋** (Part 4) 돌려 회귀 확인
- [ ] **모델별 프롬프트 분리** — `PROMPT_FOR_HAIKU`, `PROMPT_FOR_OPUS`
- [ ] 사용자 입력은 **경계 표시**(`<user_query>...</user_query>`) 로 주입 방지
- [ ] `system` 프롬프트의 토큰 수를 **주기적 모니터링**
- [ ] **PII 마스킹**: 사용자 입력에서 개인정보 제거·익명화 후 프롬프트에 삽입

---

**다음 챕터** → Ch 6. 구조화 출력

프롬프트로 JSON 형식을 **부탁**하는 건 70~90% 적중. 나머지 10~30%를 막는 **JSON Schema·Pydantic 방식**을 다음에.